In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from matplotlib.patches import Patch

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results_figures/'

# Order of all possible bins for plotting
order = [
    # '(0, 10]',
         '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]']

### Combined plot

In [ ]:
df = pd.read_parquet(os.path.join(data_dir, 'area_observed_counts.parquet'))
df = df.rename(columns={'pct_reaches_valide_obs': 'pct_reaches_valid_obs'})

df['Bin'] = pd.Categorical(df['Bin'], categories=order, ordered=True)

In [ ]:
import matplotlib.ticker as mtick  # 1. Import the ticker module

fig, ax = plt.subplots(figsize=(19,9))

palette = ['silver',
           sns.color_palette('tab20b')[2],
           sns.color_palette('tab20b')[1],]

sns.set_style('whitegrid')

sns.barplot(ax=ax, data=df,
            x='Bin', y='pct_reaches_total',
            color='silver',
                        alpha=0.7,
            legend=False, order=order)

sns.barplot(ax=ax, data=df,
            x='Bin', y='pct_reaches_valid_obs',
            color=sns.color_palette('tab20b')[2],
            alpha=0.7,
            legend=False, order=order)

sns.barplot(ax=ax, data=df,
            x='Bin', y='pct_reaches_gte_20',
            color=sns.color_palette('tab20b')[1],
            legend=False, order=order)


# Axis labels and formatting
plt.xlabel('River width (m)', fontsize=18, fontweight='bold')
plt.ylabel('Percentage of reaches', fontsize=18, fontweight='bold')
plt.xticks(rotation=-45, fontsize=16, fontweight='semibold')
plt.yticks(fontsize=16, fontweight='semibold')

# 2. Format y-axis ticks to display as percentages
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))

# Create a mapping of Bin to total_unique_reaches
counts_map = df.set_index('Bin')['total_unique_reaches'].to_dict()

# Place text just above the 100% line
y_pos = 102 

for i, bin_name in enumerate(order):
    if bin_name in counts_map:
        count_val = counts_map[bin_name]
        
        if not pd.isna(count_val):
            ax.text(i, y_pos, f'{int(count_val):,}', 
                    ha='center', va='bottom', fontsize=16, 
                    fontweight='bold', color='dimgray')

# Legend setup
legend_labels = ['Insufficient data',
                 'At least 5 pixels, at least once',
                 'At least 5 pixels, at least 20 times']
handles = [Patch(facecolor=color, label=label) for color, label in zip(palette, legend_labels)]
plt.legend(handles, legend_labels,
           # title='Region',
           framealpha=0.95,
           fontsize=16, title_fontsize=18, loc='lower right')

# Enable only horizontal grid lines
ax.yaxis.grid(True)
# ax.xaxis.grid(False)

# Optional: ensure they are behind the bars
ax.set_axisbelow(True)
# ax.set_yscale('log')

plt.ylim(0,100)

plt.savefig(f'{save_dir}reach_counts.png', dpi=300, bbox_inches='tight')

## Region plot

In [ ]:
palette_letter_dict = {
    'NE': sns.color_palette('tab20c')[1],
    'SE': sns.color_palette('tab20c')[4],
    'SW': sns.color_palette('tab20b')[9],
    # 'NW': sns.color_palette('tab20b')[-3]
    'NW': '#4C6AD1'
}

In [ ]:
df_ne = pd.read_parquet(os.path.join(data_dir, 'area_observed_counts_ne.parquet'))
df_se = pd.read_parquet(os.path.join(data_dir, 'area_observed_counts_se.parquet'))
df_sw = pd.read_parquet(os.path.join(data_dir, 'area_observed_counts_sw.parquet'))
df_nw = pd.read_parquet(os.path.join(data_dir, 'area_observed_counts_nw.parquet'))

In [ ]:
df_ne['Region'] = 'NE'
df_se['Region'] = 'SE'
df_sw['Region'] = 'SW'
df_nw['Region'] = 'NW'

In [ ]:
df_region = pd.concat([df_ne, df_se, df_sw, df_nw])
df_region['Bin'] = pd.Categorical(df_region['Bin'], categories=order, ordered=True)

In [ ]:
import matplotlib.ticker as mtick  # 1. Import the ticker module

fig, ax = plt.subplots(figsize=(19,9))

sns.set_style('whitegrid')


sns.barplot(ax=ax, data=df_region,
            x='Bin', y='pct_reaches_gte_20',
            hue='Region', palette= palette_letter_dict, 
            legend=True, order=order)

# Axis labels and formatting
plt.xlabel('River width (m)', fontsize=18, fontweight='bold')
plt.ylabel('Percentage of reaches', fontsize=18, fontweight='bold')
plt.xticks(rotation=-45, fontsize=16, fontweight='semibold')
plt.yticks(fontsize=16, fontweight='semibold')

# 2. Format y-axis ticks to display as percentages
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
plt.legend(title='Region',
           framealpha=0.95,
           fontsize=16, title_fontsize=18, loc='upper left')

# Enable only horizontal grid lines
ax.yaxis.grid(True)
# ax.xaxis.grid(False)

# Optional: ensure they are behind the bars
ax.set_axisbelow(True)
# ax.set_yscale('log')

plt.ylim(0,100)

plt.savefig(f'{save_dir}reach_counts_region.png', dpi=300, bbox_inches='tight')